# 6. Downsample clustering

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import cooler
import anndata
import scanpy as sc
import scanpy.external as sce
from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.mcds import MCDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# mpl.use('agg')
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{REF_ROOT}/wmb/'
outdir = f'{ENTEX_ROOT}/analysis/method/'


In [ ]:
adata = anndata.read_h5ad(f'{outdir}cell_301626_5kCG.h5ad')
adata

In [ ]:
meta = pd.read_csv(f'{indir}CEMBA.mC.Metadata.csv.gz', header=0, index_col=0)
meta = meta.loc[adata.obs.index]
meta

In [ ]:
meta[['L1', 'L2', 'L3', 'L4', 'R1']] = meta['CellGroup'].str.split('_', expand=True).astype(str)


In [ ]:
# for i in range(2,5):
#     meta[f'L{i}'] = meta[f'L{i-1}'] + '_' + meta[f'L{i}']
# count = pd.DataFrame([meta[['L3','L4']].drop_duplicates()['L3'].value_counts(), meta['L3'].value_counts()], index=['L4','cell']).T
# count.loc[(count['L4']>1) & (count['cell']>=1000)].sort_index()

In [ ]:
ct = 'c0_c0_c0_c0'.split('_')

# selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']==ct[2]) & (meta['L4']==ct[3])
# meta.loc[selc, 'SubClass'].value_counts()

# selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']==ct[2]) & (meta['L4']!=ct[3])
# meta.loc[selc, 'SubClass'].value_counts()

selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']!=ct[2])
meta.loc[selc, 'SubClass'].value_counts()


In [ ]:
cell_list = []
ct = 'c0_c0_c0_c0'.split('_')

selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']==ct[2]) & (meta['L4']==ct[3])
cell_list.append(np.random.choice(meta.index[selc], 1000, False))
print(selc.sum())

selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']==ct[2]) & (meta['L4']!=ct[3])
cell_list.append(np.random.choice(meta.index[selc], 1000, False))
print(selc.sum())

selc = (meta['L1']==ct[0]) & (meta['L2']==ct[1]) & (meta['L3']!=ct[2])
cell_list.append(np.random.choice(meta.index[selc], 1000, False))
print(selc.sum())

selc = (meta['L1']==ct[0]) & (meta['L2']!=ct[1])
cell_list.append(np.random.choice(meta.index[selc], 1000, False))
print(selc.sum())

selc = (meta['L1']!=ct[0])
cell_list.append(np.random.choice(meta.index[selc], 1000, False))
print(selc.sum())


In [ ]:
print(meta[['L1','L2','L3','L4']].value_counts().shape, 
      meta[['L1','L2','L3']].value_counts().shape, 
      meta[['L1','L2']].value_counts().shape, 
      meta['L1'].value_counts().shape)

In [ ]:
for i,selc in enumerate(cell_list):
    print(adata.obs.index.isin(selc).sum())
    

In [ ]:
for i,selc in enumerate(cell_list):
    tmp = adata[adata.obs.index.isin(selc)]
    tmp.write_h5ad(f'{outdir}cell_dsgroup{i}_5kCG.h5ad')


In [ ]:
adata = anndata.read_h5ad(f'{outdir}cell_dsgroup0_5kCG.h5ad')
for i,ncell in enumerate([10, 30, 50, 100, 300, 500, 1000][::-1]):
    tmp = adata[np.random.choice(adata.obs.index, ncell, False)]
    tmp.write_h5ad(f'{outdir}cell_dsgroup0_{i}_5kCG.h5ad')
    

In [ ]:
adata_list_ref = [anndata.read_h5ad(f'{outdir}cell_dsgroup{i}_5kCG.h5ad') for i in range(4)]
adata_list_alt = [anndata.read_h5ad(f'{outdir}cell_dsgroup0_{i}_5kCG.h5ad') for i in range(7)]


In [ ]:
adata_list_alt[0].X

In [ ]:
import sys
import anndata
import scanpy as sc
from ALLCools.clustering import *
from ALLCools.plot import *
from sklearn.preprocessing import normalize

npc = 20
raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)
i, j = int(sys.argv[0]), int(sys.argv[1])
adata_ref = anndata.read_h5ad(f'{outdir}cell_dsgroup{i}_5kCG.h5ad')
adata_alt = anndata.read_h5ad(f'{outdir}cell_dsgroup0_{i}_5kCG.h5ad')
adata_merge = anndata.concat([adata_ref, adata_alt], axis=0)
print(i, j, adata_merge.shape)
filter_regions(adata_merge, n_cell=5)
model.fit_transform(adata_merge, obsm_name=raw_key)
significant_pc_test(adata_merge, p_cutoff=0.05, obsm=raw_key, update=False)
adata_merge.obsm[obsm_key] = normalize(adata_merge.obsm[raw_key][:, :npc], axis=1)
tsne(adata_merge, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata_merge.obsm[f'5kCG_u{npc}_tsne'] = adata_merge.obsm['X_tsne'].copy()
sc.pp.neighbors(adata_merge, use_rep=obsm_key, n_neighbors=25)
sc.tl.leiden(adata_merge, resolution=1.0, key_added=f'5kCG_u{npc}_leiden')
adata_merge = anndata.AnnData(obs=adata_merge.obs, obsm=adata_merge.obsm, obsp=adata_merge.obsp)
adata_merge.write_h5ad(f'{outdir}cell_dsgroup{i}_{j}_5kCG_embed.h5ad')


In [ ]:
np.savetxt(f'{outdir}params.txt', [[i+1,j] for i in range(4) for j in range(7)], fmt='%d')


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
pr_score, roc_score, tpr, prc = np.zeros((4,4,7,5))
for i in range(4):
    for j in range(7):
        adata_merge = anndata.read_h5ad(f'{outdir}cell_dsgroup{i+1}_{j}_5kCG_embed.h5ad')
        ncell = adata_merge.shape[0]
        label = np.zeros(ncell, dtype=int)
        npos = len(label) - 1000
        label[1000:] = 1
        for k,npc in enumerate([5,10,15,20,25]):
            adata_merge.obs[f'5kCG_u{npc}_leiden'] = adata_merge.obs[f'5kCG_u{npc}_leiden'].astype(str)
            nct = adata_merge.obs[f'5kCG_u{npc}_leiden'].astype(str).nunique()
            count = adata_merge.obs.loc[label==1, f'5kCG_u{npc}_leiden'].value_counts() / adata_merge.obs[f'5kCG_u{npc}_leiden'].value_counts() / (npos / ncell)
            selct = count.index[count>1]
            pred = adata_merge.obs['5kCG_u20_leiden'].isin(selct).astype(int)
            # tp = (pred & label).sum()
            # tpr[i,j] = tp / npos
            # prc[i,j] = tp / pred.sum()
            pr_score[i,j,k] = average_precision_score(label, pred)
            roc_score[i,j,k] = roc_auc_score(label, pred)
        # roc_tmp = [roc_auc_score(label, adata_merge.obs[f'5kCG_u20_leiden']==str(k)) for k in range(nct)]
        # # pred = np.
        # pr_score[i,j] = np.max([average_precision_score(label, adata_merge.obs[f'5kCG_u20_leiden']==str(k)) for k in range(nct)])
        # roc_score[i,j] = np.max(roc_tmp)


In [ ]:
idx = np.array([10,30,50,100,300,500,1000])
fig, axes = plt.subplots(1, 2, figsize=(5,2), dpi=300, sharex='all')
ax = axes[0]
ax.plot(idx, [xx/(xx+1000) for xx in idx], 'k', label='Exp')
for i in range(4):
    ax.plot(idx, pr_score[i].max(axis=1)[::-1], label=f'L{4-i}')
    
ax.set_ylabel('Precision')
ax.set_xlabel('# cells')

ax = axes[1]
ax.plot(idx, [0.5 for xx in idx], 'k', label='Exp')
for i in range(4):
    ax.plot(idx, roc_score[i].max(axis=1)[::-1], label=f'L{4-i}')    
ax.set_ylabel('AUROC')
ax.set_xlabel('# cells')

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
# ax = axes[1,0]
# for i in range(4):
#     ax.plot(idx, tpr[i][::-1], label=f'Distinguished group {i}')    
# ax.plot(idx, [0.5 for xx in idx], 'k')
# ax = axes[1,1]
# for i in range(4):
#     ax.plot(idx, prc[i][::-1], label=f'Distinguished group {i}')    
# ax.plot(idx, [xx/(xx+1000) for xx in idx], 'k')
# ax.set_xticks(idx)
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(f'{outdir}cell_dsgroup_precision_roc.pdf', transparent=True)


In [ ]:
idx = np.array([10,30,50,100,300,500,1000])
fig, axes = plt.subplots(1, 2, figsize=(5,2), dpi=300, sharex='all')
ax = axes[0]
ax.plot(idx, [xx/(xx+1000) for xx in idx], 'k', label='Exp')
for i in range(4):
    ax.plot(idx, pr_score[i][::-1], label=f'L{4-i}')
    
ax.set_ylabel('Precision')
ax.set_xlabel('# cells')

ax = axes[1]
ax.plot(idx, [0.5 for xx in idx], 'k', label='Exp')
for i in range(4):
    ax.plot(idx, roc_score[i][::-1], label=f'L{4-i}')    
ax.set_ylabel('AUROC')
ax.set_xlabel('# cells')

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
# ax = axes[1,0]
# for i in range(4):
#     ax.plot(idx, tpr[i][::-1], label=f'Distinguished group {i}')    
# ax.plot(idx, [0.5 for xx in idx], 'k')
# ax = axes[1,1]
# for i in range(4):
#     ax.plot(idx, prc[i][::-1], label=f'Distinguished group {i}')    
# ax.plot(idx, [xx/(xx+1000) for xx in idx], 'k')
# ax.set_xticks(idx)
ax.set_xscale('log')
fig.tight_layout()
fig.savefig(f'{outdir}cell_dsgroup_precision_roc.pdf', transparent=True)
